# CV 4대 태스크 실습 — ④추적 실습1: MeanShift 추적

### 셀 1 — 패키지 설치 및 한글 폰트 설정

In [ ]:
!pip install -q opencv-python-headless matplotlib numpy
!apt-get -qq install -y fonts-nanum > /dev/null 2>&1

import abc
import os
import urllib.request
from dataclasses import dataclass, field
from typing import List, Sequence, Tuple

import cv2
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import numpy as np

nanum_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if os.path.exists(nanum_path):
    fm.fontManager.addfont(nanum_path)
    plt.rcParams["font.family"] = fm.FontProperties(fname=nanum_path).get_name()
plt.rcParams["axes.unicode_minus"] = False

# ============================================================================
# 이 파일은 Clean Architecture의 4개 계층으로 구성됩니다.
# (Colab 노트북 변환을 위해 "# Colab 셀 N" 마커는 그대로 유지하되,
#  그 마커들이 속한 코드 블록은 아래 계층 구분 주석을 기준으로 배치합니다.)
#   1) 도메인 계층      : 추적 결과를 표현하는 값 객체 + 추적기 추상 인터페이스
#   2) 유스케이스 계층   : "여러 프레임에 걸쳐 추적을 수행하고 경로를 쌓는다"는
#                         순수 절차 — cv2를 전혀 알지 못함(DIP)
#   3) 인프라 계층      : cv2.meanShift를 사용하는 실제 추적기 구현체
#   4) 구성 루트        : Colab에서 실제로 실행되는 코드(영상 입출력, 시각화)
# ============================================================================


# ===== [도메인 계층 / Domain Layer] =====
# 외부 라이브러리(cv2)에 의존하지 않는 순수 타입/인터페이스만 둔다.
# "추적"이라는 개념을 MeanShift 구현 세부사항과 분리해 표현한다.

@dataclass
class TrackedRegion:
    """한 프레임에서 추적된 영역(윈도우) 하나를 나타내는 값 객체.

    MeanShift는 "영역 하나"를 통째로 추적하므로, update() 호출마다
    TrackedRegion을 정확히 1개 담은 리스트를 반환하도록 설계한다.
    (실습2의 Optical Flow는 여러 개의 TrackedPoint를 반환 — LSP 관련 설계
    판단은 ObjectTracker 인터페이스의 docstring 및 README 참고)
    """

    x: int
    y: int
    w: int
    h: int

    @property
    def center(self) -> Tuple[int, int]:
        return (self.x + self.w // 2, self.y + self.h // 2)

    @property
    def as_rect(self) -> Tuple[int, int, int, int]:
        return (self.x, self.y, self.w, self.h)


@dataclass
class TrackState:
    """추적 전체 과정에서 누적되는 상태(궤적) — 유스케이스 계층이 소유/갱신한다."""

    trajectory: List[Tuple[int, int]] = field(default_factory=list)
    regions_history: List[TrackedRegion] = field(default_factory=list)

    def add(self, region: TrackedRegion) -> None:
        self.regions_history.append(region)
        self.trajectory.append(region.center)


class ObjectTracker(abc.ABC):
    """단일 대상(또는 대상 그룹)을 프레임 간 추적하는 알고리즘의 추상 인터페이스.

    - SRP: 이 인터페이스는 오직 "추적"만 책임진다. 시각화/저장은 포함하지 않는다
      (ISP: 시각화 책임은 별도 함수/클래스로 분리 — 아래 구성 루트 참고).
    - DIP: 유스케이스 계층은 이 추상 타입에만 의존하고, cv2 기반 구현(인프라
      계층)을 전혀 알지 못한다.
    - OCP: 새로운 추적 알고리즘(CSRT, KCF, 딥러닝 트래커 등)을 추가하려면
      이 인터페이스를 구현하는 새 인프라 클래스만 추가하면 되고, 유스케이스
      계층 코드는 변경할 필요가 없다.
    - LSP 설계 노트: MeanShift는 "영역 하나"를, Optical Flow(실습2)는
      "다중 점"을 추적한다는 본질적 차이가 있다. 이 둘을 억지로 하나의
      인터페이스에 욱여넣으면(예: 항상 점 1개만 반환하게 강제) 어느 한쪽이
      거짓 구현이 되어 LSP를 위반한다. 그래서 이 예제는 "각 구현이 자신에게
      자연스러운 자료형의 리스트를 반환한다"는 더 일반적인 계약을 쓴다.
      MeanShift(본 파일)는 update()가 TrackedRegion을 정확히 1개 담은
      리스트를 반환하고, Optical Flow(실습2)는 TrackedPoint를 N개 담은
      리스트를 반환한다 — 두 구현 모두 "자신의 반환 타입 리스트"라는 동일한
      계약을 정직하게 지키므로 LSP를 만족한다.
    """

    @abc.abstractmethod
    def init(self, first_frame: np.ndarray, init_region: Tuple[int, int, int, int]) -> None:
        """첫 프레임과 초기 추적 영역(x, y, w, h)으로 추적기를 초기화한다."""
        raise NotImplementedError

    @abc.abstractmethod
    def update(self, frame: np.ndarray) -> Sequence[TrackedRegion]:
        """다음 프레임에서 추적 결과를 계산해 반환한다."""
        raise NotImplementedError


# ===== [유스케이스 계층 / Use Case Layer] =====
# "비디오를 프레임 단위로 읽으며 추적기를 호출하고 결과를 누적한다"는
# 순수 오케스트레이션 로직. cv2는 import하지 않는다(DIP).

class TrackingUseCase:
    """단일 대상 추적 유스케이스.

    - SRP: 오직 "프레임 반복 + 추적 호출 + 상태 누적"만 책임진다.
      실제 추적 알고리즘(cv2.meanShift)이나 시각화 로직은 전혀 모른다.
    - DIP: 생성자로 ObjectTracker 추상 타입만 주입받는다(의존성 역전).
    """

    def __init__(self, tracker: ObjectTracker):
        self._tracker = tracker

    def run(
        self,
        first_frame: np.ndarray,
        init_region: Tuple[int, int, int, int],
        frame_reader,
        n_frames: int,
    ) -> TrackState:
        """추적을 실행하고 프레임별 결과를 TrackState에 누적해 반환한다.

        frame_reader: 호출할 때마다 (성공여부, 다음 프레임)을 돌려주는 콜러블.
        비디오 I/O 자체는 구성 루트(인프라)의 책임이며, 유스케이스는
        "프레임을 어떻게 읽는지" 알 필요가 없다(추상화된 콜백만 받는다).
        """
        self._tracker.init(first_frame, init_region)

        x, y, w, h = init_region
        state = TrackState()
        state.add(TrackedRegion(x, y, w, h))

        frames_seen: List[Tuple[int, np.ndarray, TrackedRegion]] = []
        for i in range(n_frames):
            ok, frame = frame_reader()
            if not ok:
                break
            regions = self._tracker.update(frame)
            # MeanShift는 항상 길이 1의 리스트를 반환한다(LSP 설계 노트 참고)
            region = regions[0]
            state.add(region)
            frames_seen.append((i, frame, region))

        self.frames_seen = frames_seen  # 시각화 계층(구성 루트)에서 사용
        return state


# ===== [인프라/어댑터 계층 / Infrastructure Layer] =====
# cv2를 직접 호출하는 유일한 계층. ObjectTracker 인터페이스를 구현한다.

class MeanShiftTracker(ObjectTracker):
    """OpenCV MeanShift 기반 단일 영역(윈도우) 추적기.

    색 히스토그램 역투영(back-projection) 결과에서 "확률이 가장 높은
    방향"으로 추적 윈도우를 반복 이동시킨다. ObjectTracker의 구체 구현이며,
    OCP에 따라 이 클래스를 추가/교체해도 TrackingUseCase는 수정할 필요가 없다.
    """

    def __init__(self):
        self._track_window: Tuple[int, int, int, int] = (0, 0, 0, 0)
        self._roi_hist: np.ndarray = None
        # MeanShift 종료 조건: 최대 10회 반복 또는 중심 이동이 1픽셀 미만이면 멈춤
        self._term_crit = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 1)

    def init(self, first_frame: np.ndarray, init_region: Tuple[int, int, int, int]) -> None:
        # 실제 서비스에서는 HOG/딥러닝 탐지기가 1번 프레임에서 찾아준 박스를
        # 추적 시작점으로 쓴다. 여기서는 보행자 실습(탐지 실습2)과 동일한
        # 위치를 고정 좌표로 지정해 재현성을 확보한다.
        self._track_window = init_region
        x, y, w, h = init_region
        roi = first_frame[y:y + h, x:x + w]
        hsv_roi = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
        # 그림자/저채도 영역을 제외한 마스크 (조명 변화에 덜 민감하게)
        mask_roi = cv2.inRange(hsv_roi, (0, 30, 30), (180, 255, 255))
        roi_hist = cv2.calcHist([hsv_roi], [0], mask_roi, [180], [0, 180])
        cv2.normalize(roi_hist, roi_hist, 0, 255, cv2.NORM_MINMAX)
        self._roi_hist = roi_hist

    def update(self, frame: np.ndarray) -> Sequence[TrackedRegion]:
        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        back_proj = cv2.calcBackProject([hsv], [0], self._roi_hist, [0, 180], 1)
        # MeanShift: back_proj(확률 맵)에서 추적 윈도우를 확률이 가장 높은 쪽으로 반복 이동
        _, self._track_window = cv2.meanShift(back_proj, self._track_window, self._term_crit)
        x, y, w, h = self._track_window
        return [TrackedRegion(x, y, w, h)]


# ===== [구성 루트 / Composition Root — Colab 실행 코드] =====
# 영상 입출력, 구체 추적기 생성, 유스케이스 주입, 결과 시각화/저장.
# 이 계층만 파일 I/O, matplotlib, !pip 설치를 수행한다.


### 셀 2 — 샘플 영상 준비 (탐지 실습2와 동일한 OpenCV 공식 보행자 영상)

In [ ]:
VIDEO_PATH = "/content/vtest.avi"
if not os.path.exists(VIDEO_PATH):
    url = "https://raw.githubusercontent.com/opencv/opencv/master/samples/data/vtest.avi"
    urllib.request.urlretrieve(url, VIDEO_PATH)

cap = cv2.VideoCapture(VIDEO_PATH)
cap.set(cv2.CAP_PROP_POS_FRAMES, 25)
ok, first_frame = cap.read()
if not ok:
    raise RuntimeError("영상을 열 수 없습니다.")


### 셀 3 — 추적할 대상(ROI) 선정 — 탐지(Detection) 결과를 "초기 위치"로 사용

In [ ]:
INIT_REGION = (660, 255, 40, 100)  # (x, y, w, h) — 빨간/남색 재킷을 입은 보행자(색이 뚜렷해 추적에 유리)
N_FRAMES = 40

# 구체 어댑터(인프라) 생성 → 유스케이스에 추상 타입으로 주입(DIP의 실제 적용 지점)
tracker: ObjectTracker = MeanShiftTracker()
use_case = TrackingUseCase(tracker)


### 셀 4 — MeanShift로 여러 프레임에 걸쳐 추적 — 매 프레임마다 "색 분포가 가장 비슷한 위치"로 윈도우를 이동

In [ ]:
state = use_case.run(
    first_frame=first_frame,
    init_region=INIT_REGION,
    frame_reader=lambda: cap.read(),
    n_frames=N_FRAMES,
)
cap.release()
print(f"{len(state.trajectory)}개 프레임에 걸쳐 추적 완료")

# 시각화에 쓸 프레임 선별(SRP: 추적 로직과 분리된 순수 시각화 보조 함수)
frames_to_draw: List[Tuple[int, np.ndarray]] = []
for i, frame, region in use_case.frames_seen:
    if i % 8 == 0 or i == len(use_case.frames_seen) - 1:
        vis = frame.copy()
        x, y, w, h = region.as_rect
        cv2.rectangle(vis, (x, y), (x + w, y + h), (122, 92, 158), 2)
        frames_to_draw.append((i, vis.copy()))


### 셀 5 — 이동 경로(trajectory) 오버레이 — 탐지(공간)만으로는 알 수 없는 "시간에 따른 이동"을 보여준다

In [ ]:
final_vis = frames_to_draw[-1][1].copy()
for i in range(1, len(state.trajectory)):
    cv2.line(final_vis, state.trajectory[i - 1], state.trajectory[i], (0, 140, 255), 2)
for pt in state.trajectory:
    cv2.circle(final_vis, pt, 3, (0, 140, 255), -1)


### 셀 6 — 결과 시각화 — 여러 시점의 추적 박스 + 최종 이동 경로

In [ ]:
fig, axes = plt.subplots(1, len(frames_to_draw) + 1, figsize=(4 * (len(frames_to_draw) + 1), 4))
for ax, (idx, vis) in zip(axes[:-1], frames_to_draw):
    ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    ax.set_title(f"frame +{idx}")
    ax.axis("off")
axes[-1].imshow(cv2.cvtColor(final_vis, cv2.COLOR_BGR2RGB))
axes[-1].set_title("전체 이동 경로(trajectory)")
axes[-1].axis("off")
plt.suptitle("MeanShift 추적 — 같은 보행자를 프레임마다 재탐색하지 않고 계속 따라간다", fontsize=13)
plt.tight_layout()
plt.savefig("/content/meanshift_result.png", dpi=110, bbox_inches="tight")
plt.show()
print("저장 완료: /content/meanshift_result.png")
